# Т-Банк — корпоративные новости и торговые сессии

Пилотная выборка из 15 публикаций за май–август 2024 года для исследования реакции акций на корпоративные новости в период ребрендинга.

Анализ включает нормализацию времени публикаций, определение этапа торговой сессии и сопоставление новостей с дневными котировками TCSG.

Исходные данные включены в ноутбук. Зависимости — Python, pandas, NumPy, requests. Для отображения таблиц используется IPython.

## 1. Импорты и функции

Временные метки приведены к `Europe/Moscow`. В исходных публикациях время указано с точностью до минуты без UTC-смещения. Московский часовой пояс принят как допущение пилотной выборки.

Секунды `00` обозначают начало указанной минуты.

In [1]:
from datetime import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display

TZ = "Europe/Moscow"
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 30)

/Users/nadezhda/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def publication_phase(timestamp, trading_dates):
    """Положение публикации в историческом расписании, по московскому времени."""
    if timestamp.date() not in trading_dates:
        return "non_trading_date"
    clock = timestamp.time()
    if clock < time(9, 50):
        return "before_opening_auction"
    if clock < time(10):
        return "opening_auction"
    if clock < time(18, 40):
        return "main_continuous"
    if clock < time(18, 50):
        return "closing_auction"
    if clock < time(19):
        return "between_sessions"
    if clock < time(23, 50):
        return "evening_clock_admission_unchecked"
    return "after_evening"


def load_moex_history(market, board, security, date_from, date_till):
    """Загружает страницы истории MOEX ISS, включая переходы между страницами."""
    url = (
        f"https://iss.moex.com/iss/history/engines/stock/markets/{market}"
        f"/boards/{board}/securities/{security}.json"
    )
    frames = []
    offset = 0
    with requests.Session() as session:
        while True:
            response = session.get(
                url,
                params={
                    "from": date_from,
                    "till": date_till,
                    "start": offset,
                    "tradingsession": 1,
                    "iss.meta": "off",
                    "iss.only": "history,history.cursor",
                },
                timeout=(5, 20),
            )
            response.raise_for_status()
            payload = response.json()
            block = payload["history"]
            page = pd.DataFrame(block["data"], columns=block["columns"])
            if page.empty:
                break
            frames.append(page)
            cursor_block = payload["history.cursor"]
            cursor = dict(zip(cursor_block["columns"], cursor_block["data"][0]))
            offset = int(cursor["INDEX"]) + int(cursor["PAGESIZE"])
            if offset >= int(cursor["TOTAL"]):
                break

    if not frames:
        raise ValueError(f"{security}: история за период отсутствует.")
    history = pd.concat(frames, ignore_index=True)
    history["TRADEDATE"] = pd.to_datetime(history["TRADEDATE"])
    if history["TRADEDATE"].duplicated().any():
        raise ValueError("В истории обнаружено несколько строк на торговую дату.")
    if "TRADINGSESSION" in history.columns:
        if not pd.to_numeric(history["TRADINGSESSION"]).eq(1).all():
            raise ValueError("В ответ попали другие торговые сессии.")
    return history.sort_values("TRADEDATE").reset_index(drop=True)

## 2. Новости

Каждая запись содержит исходный заголовок, время публикации и ссылку на источник. Используется время над заголовком статьи. Ссылка «Есть обновление от …» относится к отдельному материалу.

`topic_manual` содержит предварительную ручную тематическую разметку. `information_group` объединяет публикации одного раскрытия для учёта зависимости между наблюдениями. Разметка тональности в текущей выборке отсутствует.

In [3]:
news_records = [
  {
    "news_id": "ifx_959599",
    "published_local": "2024-05-08 17:32",
    "headline": "Акционеры \"ТКС Холдинга\" одобрили размещение допэмиссии акций для покупки Росбанка",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/959599",
    "topic_manual": "capital_and_deals",
    "information_group": "rosbank_acquisition",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_960799",
    "published_local": "2024-05-16 12:13",
    "headline": "\"ТКС Холдинг\" в I квартале увеличил чистую прибыль по МСФО на 37%",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/960799",
    "topic_manual": "financial_results",
    "information_group": "q1_disclosure",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_960802",
    "published_local": "2024-05-16 12:21",
    "headline": "\"ТКС Холдинг\" выкупил с рынка 12 млн собственных акций на 35 млрд рублей",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/960802",
    "topic_manual": "buyback",
    "information_group": "q1_disclosure",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_960806",
    "published_local": "2024-05-16 12:34",
    "headline": "\"ТКС Холдинг\" в 2024 году ждет роста чистой прибыли более чем на 30%",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/960806",
    "topic_manual": "guidance",
    "information_group": "q1_disclosure",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_964973",
    "published_local": "2024-06-05 09:42",
    "headline": "Тинькофф банк сменит название на Т-банк",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/964973",
    "topic_manual": "rebranding",
    "information_group": "brand_announcement",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_965237",
    "published_local": "2024-06-06 10:12",
    "headline": "ЦБ просит суд привлечь \"Тинькофф страхование\" к ответственности на 7,7 млрд руб.",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/965237",
    "topic_manual": "legal",
    "information_group": "insurance_claim",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_965365",
    "published_local": "2024-06-06 15:42",
    "headline": "\"ТКС Холдинг\" подтвердил план вернуться к выплате дивидендов",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/965365",
    "topic_manual": "dividends",
    "information_group": "dividend_intention",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_969130",
    "published_local": "2024-07-03 18:54",
    "headline": "Акционеры \"ТКС Холдинга\" решили не выплачивать дивиденды за 2023 год",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/969130",
    "topic_manual": "dividends",
    "information_group": "dividend_2023",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_969400",
    "published_local": "2024-07-05 09:46",
    "headline": "Тинькофф банк сменил название на ТБанк",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/969400",
    "topic_manual": "rebranding",
    "information_group": "legal_name_change",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_971927",
    "published_local": "2024-07-22 18:41",
    "headline": "\"ТКС Холдинг\" стал владельцем еще одной брокерской компании",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/971927",
    "topic_manual": "capital_and_deals",
    "information_group": "broker_acquisition",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_975924",
    "published_local": "2024-08-09 20:08",
    "headline": "ТБанк во II квартале увеличил кредитный портфель по РСБУ на 26% до 2,1 трлн рублей",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/975924",
    "topic_manual": "financial_results",
    "information_group": "bank_q2_ras",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_977190",
    "published_local": "2024-08-19 20:36",
    "headline": "Совет директоров \"ТКС Холдинга\" утвердил дивидендную политику",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/977190",
    "topic_manual": "dividends",
    "information_group": "dividend_policy",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_977240",
    "published_local": "2024-08-20 10:24",
    "headline": "\"ТКС Холдинг\" увеличил квартальную чистую прибыль по МСФО на 15%",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/977240",
    "topic_manual": "financial_results",
    "information_group": "q2_disclosure",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_977257",
    "published_local": "2024-08-20 12:26",
    "headline": "\"ТКС Холдинг\" продал Росбанку права требования по розничным кредитам на 40,3 млрд рублей",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/977257",
    "topic_manual": "capital_and_deals",
    "information_group": "q2_disclosure",
    "verified_on": "2026-11-19"
  },
  {
    "news_id": "ifx_978113",
    "published_local": "2024-08-26 12:29",
    "headline": "Чистая прибыль ТБанка за 7 месяцев по РСБУ выросла на 35%",
    "source": "Интерфакс",
    "url": "https://www.interfax.ru/business/978113",
    "topic_manual": "financial_results",
    "information_group": "bank_july_ras",
    "verified_on": "2026-11-19"
  }
]

In [4]:
news = pd.DataFrame(news_records)
news["published_at_msk"] = pd.to_datetime(
    news["published_local"], format="%Y-%m-%d %H:%M"
).dt.tz_localize(TZ)
news["published_at_utc"] = news["published_at_msk"].dt.tz_convert("UTC")
news["time_precision"] = "minute"
news["timezone_basis"] = "assumed_moscow_from_visible_publication_time"
news = news.sort_values("published_at_msk").reset_index(drop=True)

assert len(news) == 15
assert news["news_id"].is_unique
assert news["url"].is_unique
assert news["published_at_msk"].notna().all()

display(news[["news_id", "published_at_msk", "headline", "topic_manual", "url"]])

,news_id,published_at_msk,headline,topic_manual,url
0,ifx_959599,2024-05-08 17:32:00+03:00,"Акционеры ""ТКС Холдинга"" одобрили размещение допэмиссии акций для покупки Росбанка",capital_and_deals,https://www.interfax.ru/business/959599
1,ifx_960799,2024-05-16 12:13:00+03:00,"""ТКС Холдинг"" в I квартале увеличил чистую прибыль по МСФО на 37%",financial_results,https://www.interfax.ru/business/960799
2,ifx_960802,2024-05-16 12:21:00+03:00,"""ТКС Холдинг"" выкупил с рынка 12 млн собственных акций на 35 млрд рублей",buyback,https://www.interfax.ru/business/960802
3,ifx_960806,2024-05-16 12:34:00+03:00,"""ТКС Холдинг"" в 2024 году ждет роста чистой прибыли более чем на 30%",guidance,https://www.interfax.ru/business/960806
4,ifx_964973,2024-06-05 09:42:00+03:00,Тинькофф банк сменит название на Т-банк,rebranding,https://www.interfax.ru/business/964973
5,ifx_965237,2024-06-06 10:12:00+03:00,"ЦБ просит суд привлечь ""Тинькофф страхование"" к ответственности на 7,7 млрд руб.",legal,https://www.interfax.ru/business/965237
6,ifx_965365,2024-06-06 15:42:00+03:00,"""ТКС Холдинг"" подтвердил план вернуться к выплате дивидендов",dividends,https://www.interfax.ru/business/965365
7,ifx_969130,2024-07-03 18:54:00+03:00,"Акционеры ""ТКС Холдинга"" решили не выплачивать дивиденды за 2023 год",dividends,https://www.interfax.ru/business/969130
8,ifx_969400,2024-07-05 09:46:00+03:00,Тинькофф банк сменил название на ТБанк,rebranding,https://www.interfax.ru/business/969400
9,ifx_971927,2024-07-22 18:41:00+03:00,"""ТКС Холдинг"" стал владельцем еще одной брокерской компании",capital_and_deals,https://www.interfax.ru/business/971927


## 3. Торговое расписание 2024 года

| Время, Москва | Этап |
|---|---|
| 09:50–10:00 | Аукцион открытия |
| 10:00–18:40 | Непрерывные торги основной сессии |
| 18:40–18:50 | Аукцион закрытия |
| 18:50–19:00 | Перерыв |
| 19:00–23:50 | Вечерний интервал для допущенных бумаг |

Источники расписания — [исторический материал MOEX](https://www.moex.com/files/45f4r2h2atbb1tff7dqmvmbqc8) и [сообщение биржи за июнь 2024 года](https://www.moex.com/n70545).

Классификация использует минутные границы исторического расписания. Внеплановые остановки торгов и индивидуальный допуск TCSG к вечерней сессии не моделируются. Метка вечернего интервала определяется по времени публикации.

## 4. Торговый календарь

Календарь за 02.05–30.08.2024 построен по датам из [истории TCSG на SwingTrading](https://swingtrading.ru/moex/stock/tcsg/history/). Отсутствующие даты исключены из сопоставления.

Проверки учитывают отсутствие торговых записей за 9 мая и 12 июня.

In [5]:
trading_date_records = [
  "2024-05-02",
  "2024-05-03",
  "2024-05-06",
  "2024-05-07",
  "2024-05-08",
  "2024-05-10",
  "2024-05-13",
  "2024-05-14",
  "2024-05-15",
  "2024-05-16",
  "2024-05-17",
  "2024-05-20",
  "2024-05-21",
  "2024-05-22",
  "2024-05-23",
  "2024-05-24",
  "2024-05-27",
  "2024-05-28",
  "2024-05-29",
  "2024-05-30",
  "2024-05-31",
  "2024-06-03",
  "2024-06-04",
  "2024-06-05",
  "2024-06-06",
  "2024-06-07",
  "2024-06-10",
  "2024-06-11",
  "2024-06-13",
  "2024-06-14",
  "2024-06-17",
  "2024-06-18",
  "2024-06-19",
  "2024-06-20",
  "2024-06-21",
  "2024-06-24",
  "2024-06-25",
  "2024-06-26",
  "2024-06-27",
  "2024-06-28",
  "2024-07-01",
  "2024-07-02",
  "2024-07-03",
  "2024-07-04",
  "2024-07-05",
  "2024-07-08",
  "2024-07-09",
  "2024-07-10",
  "2024-07-11",
  "2024-07-12",
  "2024-07-15",
  "2024-07-16",
  "2024-07-17",
  "2024-07-18",
  "2024-07-19",
  "2024-07-22",
  "2024-07-23",
  "2024-07-24",
  "2024-07-25",
  "2024-07-26",
  "2024-07-29",
  "2024-07-30",
  "2024-07-31",
  "2024-08-01",
  "2024-08-02",
  "2024-08-05",
  "2024-08-06",
  "2024-08-07",
  "2024-08-08",
  "2024-08-09",
  "2024-08-12",
  "2024-08-13",
  "2024-08-14",
  "2024-08-15",
  "2024-08-16",
  "2024-08-19",
  "2024-08-20",
  "2024-08-21",
  "2024-08-22",
  "2024-08-23",
  "2024-08-26",
  "2024-08-27",
  "2024-08-28",
  "2024-08-29",
  "2024-08-30"
]

In [6]:
calendar = pd.DataFrame({"session_date": pd.to_datetime(trading_date_records)})
calendar = calendar.sort_values("session_date").reset_index(drop=True)
calendar["main_open"] = (
    calendar["session_date"] + pd.Timedelta(hours=10)
).dt.tz_localize(TZ)
calendar["main_close"] = (
    calendar["session_date"] + pd.Timedelta(hours=18, minutes=50)
).dt.tz_localize(TZ)
trading_dates = set(calendar["session_date"].dt.date)

news["session_phase"] = news["published_at_msk"].apply(
    publication_phase, trading_dates=trading_dates
)
display(news[["published_at_msk", "headline", "session_phase"]])

,published_at_msk,headline,session_phase
0,2024-05-08 17:32:00+03:00,"Акционеры ""ТКС Холдинга"" одобрили размещение допэмиссии акций для покупки Росбанка",main_continuous
1,2024-05-16 12:13:00+03:00,"""ТКС Холдинг"" в I квартале увеличил чистую прибыль по МСФО на 37%",main_continuous
2,2024-05-16 12:21:00+03:00,"""ТКС Холдинг"" выкупил с рынка 12 млн собственных акций на 35 млрд рублей",main_continuous
3,2024-05-16 12:34:00+03:00,"""ТКС Холдинг"" в 2024 году ждет роста чистой прибыли более чем на 30%",main_continuous
4,2024-06-05 09:42:00+03:00,Тинькофф банк сменит название на Т-банк,before_opening_auction
5,2024-06-06 10:12:00+03:00,"ЦБ просит суд привлечь ""Тинькофф страхование"" к ответственности на 7,7 млрд руб.",main_continuous
6,2024-06-06 15:42:00+03:00,"""ТКС Холдинг"" подтвердил план вернуться к выплате дивидендов",main_continuous
7,2024-07-03 18:54:00+03:00,"Акционеры ""ТКС Холдинга"" решили не выплачивать дивиденды за 2023 год",between_sessions
8,2024-07-05 09:46:00+03:00,Тинькофф банк сменил название на ТБанк,before_opening_auction
9,2024-07-22 18:41:00+03:00,"""ТКС Холдинг"" стал владельцем еще одной брокерской компании",closing_auction


## 5. Привязка к основной сессии

Для каждой публикации определяются два временных ориентира.

- `first_close_date` — дата первого закрытия основной сессии после публикации.
- `first_full_session_date` — дата первой основной сессии, целиком расположенной после публикации.

Окно полной сессии характеризует последующее движение цены и может пропускать часть немедленной реакции. Оценка реакции непосредственно после дневной публикации требует внутридневных котировок.

Доходность от предыдущего закрытия включает движение до публикации, если новость вышла во время торгов.

Конец минуты публикации принят за консервативную границу доступности информации.

In [7]:
matched = news.sort_values("published_at_msk").copy()
matched["publication_minute_end"] = (
    matched["published_at_msk"] + pd.Timedelta(minutes=1)
)

close_targets = calendar[["session_date", "main_close"]].rename(columns={
    "session_date": "first_close_date",
    "main_close": "first_main_close_after",
})
matched = pd.merge_asof(
    matched, close_targets,
    left_on="publication_minute_end",
    right_on="first_main_close_after",
    direction="forward",
)

full_targets = calendar[["session_date", "main_open", "main_close"]].rename(columns={
    "session_date": "first_full_session_date",
    "main_open": "full_session_open",
    "main_close": "full_session_close",
})
matched = pd.merge_asof(
    matched, full_targets,
    left_on="publication_minute_end",
    right_on="full_session_open",
    direction="forward",
)

display(matched[[
    "news_id", "published_at_msk", "session_phase",
    "first_close_date", "first_full_session_date",
]])

,news_id,published_at_msk,session_phase,first_close_date,first_full_session_date
0,ifx_959599,2024-05-08 17:32:00+03:00,main_continuous,2024-05-08,2024-05-10
1,ifx_960799,2024-05-16 12:13:00+03:00,main_continuous,2024-05-16,2024-05-17
2,ifx_960802,2024-05-16 12:21:00+03:00,main_continuous,2024-05-16,2024-05-17
3,ifx_960806,2024-05-16 12:34:00+03:00,main_continuous,2024-05-16,2024-05-17
4,ifx_964973,2024-06-05 09:42:00+03:00,before_opening_auction,2024-06-05,2024-06-05
5,ifx_965237,2024-06-06 10:12:00+03:00,main_continuous,2024-06-06,2024-06-07
6,ifx_965365,2024-06-06 15:42:00+03:00,main_continuous,2024-06-06,2024-06-07
7,ifx_969130,2024-07-03 18:54:00+03:00,between_sessions,2024-07-04,2024-07-04
8,ifx_969400,2024-07-05 09:46:00+03:00,before_opening_auction,2024-07-05,2024-07-05
9,ifx_971927,2024-07-22 18:41:00+03:00,closing_auction,2024-07-22,2024-07-23


## 6. Сопоставление с дневными котировками

Фрагмент дневной истории TCSG содержит 20 записей цен открытия и закрытия в рублях. Источник — [SwingTrading](https://swingtrading.ru/moex/stock/tcsg/history/).

Состав торговых сессий в дневных ценах источника не установлен. Поле `quote_session_status` сохраняет этот статус. Таблица используется для проверки соединения по дате первой полной сессии.

Оценка доходности относительно рынка требует согласованных по времени котировок TCSG и значений IMOEX.

In [8]:
quote_records = [
  {
    "date": "2024-05-08",
    "open": 3148.5,
    "close": 3086
  },
  {
    "date": "2024-05-10",
    "open": 3088.5,
    "close": 3100
  },
  {
    "date": "2024-05-16",
    "open": 3105,
    "close": 3089.5
  },
  {
    "date": "2024-05-17",
    "open": 3093,
    "close": 3060
  },
  {
    "date": "2024-06-04",
    "open": 2805,
    "close": 2893
  },
  {
    "date": "2024-06-05",
    "open": 2905,
    "close": 2919
  },
  {
    "date": "2024-06-06",
    "open": 2917,
    "close": 2912.5
  },
  {
    "date": "2024-06-07",
    "open": 2915.5,
    "close": 2957
  },
  {
    "date": "2024-07-03",
    "open": 2945,
    "close": 2903.5
  },
  {
    "date": "2024-07-04",
    "open": 2903,
    "close": 2831.5
  },
  {
    "date": "2024-07-05",
    "open": 2832,
    "close": 2815.5
  },
  {
    "date": "2024-07-22",
    "open": 2773.5,
    "close": 2772.5
  },
  {
    "date": "2024-07-23",
    "open": 2772.5,
    "close": 2729
  },
  {
    "date": "2024-08-09",
    "open": 2575,
    "close": 2572
  },
  {
    "date": "2024-08-12",
    "open": 2572,
    "close": 2617.5
  },
  {
    "date": "2024-08-19",
    "open": 2585,
    "close": 2615.5
  },
  {
    "date": "2024-08-20",
    "open": 2621,
    "close": 2617
  },
  {
    "date": "2024-08-21",
    "open": 2622,
    "close": 2630
  },
  {
    "date": "2024-08-26",
    "open": 2589,
    "close": 2576
  },
  {
    "date": "2024-08-27",
    "open": 2581,
    "close": 2546.5
  }
]

In [9]:
quotes = pd.DataFrame(quote_records).rename(columns={
    "date": "quote_date",
    "open": "open_reference_rub",
    "close": "close_reference_rub",
})
quotes["quote_date"] = pd.to_datetime(quotes["quote_date"])
quotes["quote_source"] = "https://swingtrading.ru/moex/stock/tcsg/history/"
quotes["quote_session_status"] = "unverified_daily_aggregation"

matched = matched.merge(
    quotes,
    left_on="first_full_session_date",
    right_on="quote_date",
    how="left",
    validate="many_to_one",
)
matched["quote_join_ok"] = matched["close_reference_rub"].notna()
matched["publications_on_full_session"] = matched.groupby(
    "first_full_session_date"
)["news_id"].transform("size")

display(matched[[
    "news_id", "first_full_session_date", "open_reference_rub",
    "close_reference_rub", "quote_join_ok",
]])

,news_id,first_full_session_date,open_reference_rub,close_reference_rub,quote_join_ok
0,ifx_959599,2024-05-10,3088.5,3100.0,True
1,ifx_960799,2024-05-17,3093.0,3060.0,True
2,ifx_960802,2024-05-17,3093.0,3060.0,True
3,ifx_960806,2024-05-17,3093.0,3060.0,True
4,ifx_964973,2024-06-05,2905.0,2919.0,True
5,ifx_965237,2024-06-07,2915.5,2957.0,True
6,ifx_965365,2024-06-07,2915.5,2957.0,True
7,ifx_969130,2024-07-04,2903.0,2831.5,True
8,ifx_969400,2024-07-05,2832.0,2815.5,True
9,ifx_971927,2024-07-23,2772.5,2729.0,True


## 7. Проверки временной привязки

| Публикация | Первая полная основная сессия |
|---|---|
| Ребрендинг, 5 июня, 09:42 | 5 июня |
| Дивиденды, 3 июля, 18:54 | 4 июля |
| Покупка брокера, 22 июля, 18:41 | 23 июля |
| Кредитный портфель, пятница 9 августа, 20:08 | Понедельник 12 августа |
| Дивидендная политика, 19 августа, 20:36 | 20 августа |

Дополнительные проверки охватывают уникальность публикаций, покрытие котировками и положение выбранных сессий относительно времени новости.

Группировка по дате полной сессии выявляет пересечения окон. В частности, 20 августа опубликованы квартальные результаты и сообщение о продаже кредитных требований.

In [10]:
checks = matched.set_index("news_id")
expected_dates = {
    "ifx_964973": "2024-06-05",
    "ifx_969130": "2024-07-04",
    "ifx_971927": "2024-07-23",
    "ifx_975924": "2024-08-12",
    "ifx_977190": "2024-08-20",
}
for news_id, expected_date in expected_dates.items():
    assert checks.loc[news_id, "first_full_session_date"] == pd.Timestamp(expected_date)

assert (matched["full_session_open"] >= matched["publication_minute_end"]).all()
assert (matched["first_main_close_after"] >= matched["publication_minute_end"]).all()
assert matched["quote_join_ok"].all()
assert pd.Timestamp("2024-05-09").date() not in trading_dates
assert pd.Timestamp("2024-06-12").date() not in trading_dates

display(pd.DataFrame({
    "Проверка": [
        "Публикации",
        "Информационные группы",
        "Строки с присоединённой дневной ценой",
    ],
    "Количество": [
        len(news),
        news["information_group"].nunique(),
        int(matched["quote_join_ok"].sum()),
    ],
}))
display(matched.groupby("first_full_session_date").agg(
    publications=("news_id", "size"),
    information_groups=("information_group", "nunique"),
))

,Проверка,Количество
0,Публикации,15
1,Информационные группы,12
2,Строки с присоединённой дневной ценой,15


,publications,information_groups
first_full_session_date,,
2024-05-10,1,1
2024-05-17,3,1
2024-06-05,1,1
2024-06-07,2,2
2024-07-04,1,1
2024-07-05,1,1
2024-07-23,1,1
2024-08-12,1,1
2024-08-20,1,1


## 8. Наблюдения и ограничения

Объявление нового бренда 5 июня и юридическое переименование в июле представлены как отдельные события.

Три публикации 16 мая связаны с одним раскрытием. Две публикации 20 августа также относятся к общему информационному пакету. Число статей превышает число независимых событий.

На следующий день после объявления бренда опубликованы сообщения об иске к страховщику и планах по дивидендам. Многодневное окно вокруг ребрендинга включает эти события.

Выборка сформирована вручную из одного издания для проверки временной привязки. Её объём и способ отбора ограничивают возможность обучения NLP-моделей и статистического обобщения результатов.

## 9. Исторические данные MOEX ISS

Загрузчик получает историю TCSG в режиме TQBR и индекса IMOEX в режиме SNDX. Запросы используют пагинацию и параметр `tradingsession=1`, соответствующий основной сессии. [Описание кодов сессий MOEX](https://www.moex.com/n28895?nt=209).

При наличии поля `TRADINGSESSION` проверяется соответствие коду основной сессии. Ответы без этого поля требуют отдельной проверки состава данных.

Загрузка управляется параметром `DOWNLOAD_MOEX`. Встроенная выборка обеспечивает воспроизводимость анализа при `DOWNLOAD_MOEX=False`.

Маршруты API — [TCSG](https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/TCSG.json) и [IMOEX](https://iss.moex.com/iss/history/engines/stock/markets/index/boards/SNDX/securities/IMOEX.json).

In [11]:
DOWNLOAD_MOEX = False

if DOWNLOAD_MOEX:
    official_stock = load_moex_history(
        "shares", "TQBR", "TCSG", "2024-05-02", "2024-08-30"
    )
    official_index = load_moex_history(
        "index", "SNDX", "IMOEX", "2024-05-02", "2024-08-30"
    )
    display(official_stock.head())
    display(official_index.head())
    print("История TCSG и IMOEX загружена.")

## 10. Экспорт данных

Параметр `EXPORT_TABLES` управляет сохранением таблиц в `data/processed` относительно рабочей директории Jupyter.

- `news_pilot.csv` — публикации и ручная тематическая разметка.
- `news_session_mapping.csv` — привязка публикаций к сессиям и дневным ценам.
- `tcsg_reference_quotes.csv` — исходный фрагмент котировок TCSG.

Кодировка CSV — UTF-8 с BOM.

In [12]:
EXPORT_TABLES = False

if EXPORT_TABLES:
    output_dir = Path("data/processed")
    output_dir.mkdir(parents=True, exist_ok=True)
    news.to_csv(output_dir / "news_pilot.csv", index=False, encoding="utf-8-sig")
    matched.to_csv(output_dir / "news_session_mapping.csv", index=False, encoding="utf-8-sig")
    quotes.to_csv(output_dir / "tcsg_reference_quotes.csv", index=False, encoding="utf-8-sig")
    print(f"Таблицы сохранены в {output_dir.resolve()}")